# Qwen на экзаменационном наборе

По умолчанию проверяем и восстанавливаем исходный JSONL с побайтным совпадением. Чтобы запустить модель с нуля, включи GPU и Internet в Kaggle, установи версии из README и поставь `RERUN=True`. Новый файл сохраняется в отдельной папке и сравнивается с опубликованным. Исторический GPU-прогон с выводами записан в [оригинальном ноутбуке](../kaggle/production_original.ipynb).

In [1]:
import sys
from pathlib import Path
from IPython.display import display

cwd = Path.cwd().resolve()
candidates = (cwd, *cwd.parents, Path("/kaggle/working/data-labeling-shad"))
ROOT = next((p for p in candidates if (p / "model_comparison/workflow.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Открой ноутбук из клона репозитория или склонируй его в /kaggle/working/data-labeling-shad")
sys.path.insert(0, str(ROOT))
print("Корень репозитория найден")

Корень репозитория найден


In [2]:
from model_comparison.run_production import replay_submission, run_production
from model_comparison.workflow import PUBLISHED, PUBLISHED_SHA256

RERUN = False
DATASET = ROOT / "data/production_dataset.jsonl"
restore_dir = ROOT / "runs/submission_replay"
rerun_dir = ROOT / "runs/production_qwen"
print("Режим:", "новый инференс" if RERUN else "побайтное восстановление")

Режим: побайтное восстановление


In [3]:
if RERUN:
    submission = run_production(DATASET, rerun_dir)
else:
    submission = replay_submission(restore_dir, DATASET)

print(f"{submission.name}: {submission.stat().st_size:,} байт")
print("Совпадает с опубликованным файлом:", submission.read_bytes() == PUBLISHED.read_bytes())
print("SHA-256 опубликованного файла:", PUBLISHED_SHA256)

submission.jsonl: 37,551 байт
Совпадает с опубликованным файлом: True
SHA-256 опубликованного файла: 093617e7c28c2e20def000d9b8e2c8b77953707b9f46235f33d56e9c03eaa0ce
